# Non-Causal Transformer Encoder — Travel Time Model

A fourth model on the same per-stop travel-time regression task: a standard **bidirectional
Transformer encoder** (embeddings → positional encoding → stacked self-attention/FFN blocks →
MLP head), with no causal mask.

## Why non-causal, and when this is (and isn't) valid
The LSTM, Conv-Attn-LSTM, and TFT notebooks all enforce causality — a prediction at stop *t* can
only use information from stops `<= t`, because those models are framed as if predicting travel
time in real time as the bus moves along the route. This notebook drops that constraint: every
stop can attend to every other stop in the trip, forward and backward.

**That only makes sense for offline analysis** — e.g. post-hoc studying which segments were
anomalously slow, imputing/smoothing a completed trip's travel times, or as an upper-bound
reference point ("how good could a model get if it could see the whole trip"). It is **not** a
valid architecture for live ETA prediction, since it uses information (later stops' actual
conditions) that wouldn't exist yet in a real-time setting. If you later want a causal version of
this same encoder (e.g. to compare non-causal vs. causal attention head-to-head), it's a one-line
change: reintroduce the upper-triangular mask from the Conv-Attn-LSTM notebook when building
`attn_mask`.

## Why not Informer / PatchTST
Both are built to make attention sub-quadratic for *long* sequences (thousands of steps) via
sparse/ProbSparse attention (Informer) or patch-based tokenization (PatchTST). A single bus trip
here is on the order of tens of stops, so a plain `O(T^2)` full-attention Transformer is already
cheap — the efficiency tricks those models offer wouldn't change results and would only add
complexity. If your sequences are much longer in practice (e.g. modeling a whole day of a
vehicle's stops end-to-end rather than one trip at a time), that's the point at which Informer or
PatchTST-style attention becomes worth adopting.

## Architecture
```
categorical codes ─► embeddings ─┐
                                  ├─ concat ─► Linear projection to d_model ─► + positional encoding
numeric features ─────────────────┘                                                   │
                                                                                        ▼
                                                                     N x [self-attention (full, non-causal,
                                                                          padding-masked) + FFN] encoder layers
                                                                                        │
                                                                                        ▼
                                                                          MLP head ─► per-stop travel time
```

Data loading, categorical encoding, normalization (fit on train only), sequence building,
Dataset/collate, masked MSE/MAE loss, evaluation metrics, and seeds are identical to
`train_lstm.ipynb` / `train_conv_attn_lstm.ipynb`, so all these models remain directly comparable.


In [ ]:
import math
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

# =====================================================================
# 0. Config — mirrors train_lstm.ipynb / train_conv_attn_lstm.ipynb's feature split
# =====================================================================
DATA_PATH = "raw/processed_gtfs/baseline_dataset.parquet"
TARGET = "travel_time"
SEQ_ID_COLS = ["trip_id", "start_date"]
ORDER_COL = "stop_sequence"

CATEGORICAL = ["route_id", "direction_id", "shape_id", "service_id", 'is_raining', 'is_snowing', 'is_fog', "is_weekend",
    "is_federal_holiday",
    "is_school_day",
    "has_major_event", "is_peak", 'weathercode',]
NUMERIC = [
    "stop_sequence", "trip_progress",
    "hour", "weekday", "month",
    "scheduled_arrival", "scheduled_departure", "scheduled_segment_time",
    "stop_lat", "stop_lon", "latitude", "longitude", "bearing",
    'temperature_c', 'precipitation_mm', 'snowfall_cm', 'windspeed_kmh',
    'segment_length',
    'scheduled_segment_speed_mps',
    # 'upstream_delay_seconds',
    'speed_mps',
    # 'headway_seconds',
    "ridership",
    "transfers",
]

EMB_DIM_CAP = 150     # max embedding dim per categorical column (same dynamic-sizing rule as baseline)

# --- model-specific hyperparameters ---
D_MODEL = 256          # transformer model dimension
N_HEADS = 8            # attention heads (D_MODEL must be divisible by N_HEADS)
N_LAYERS = 4           # encoder layers
DIM_FEEDFORWARD = 512  # FFN width inside each encoder layer
DROPOUT = 0.1
MAX_SEQ_LEN = 128      # cap for the positional encoding table; increase if a trip exceeds this

BATCH_SIZE = 64
LR = 1e-3
MAX_EPOCHS = 100
PATIENCE = 12
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
np.random.seed(42)


In [12]:
df = pl.read_parquet(DATA_PATH)
print(f"Loaded {df.shape}")

df = df.with_columns(
    (pl.col("trip_id") + "_" + pl.col("start_date").cast(pl.Utf8)).alias("_seq_id")
)

groups = df["_seq_id"].to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=groups))
train_df = df[train_idx]
test_df = df[test_idx]
print(f"train rows: {train_df.height:,} | test rows: {test_df.height:,}")
print(f"train trips: {train_df['_seq_id'].n_unique():,} | test trips: {test_df['_seq_id'].n_unique():,}")


Loaded (5596768, 40)
train rows: 4,478,560 | test rows: 1,118,208
train trips: 96,308 | test trips: 24,077


In [13]:
# =====================================================================
# 2. Encode categoricals — fit on TRAIN only, reserve 0 for padding/unknown
# =====================================================================
cat_maps = {}
cat_cardinalities = []
for col in CATEGORICAL:
    uniques = train_df[col].unique().sort().to_list()
    cat_maps[col] = {v: i + 1 for i, v in enumerate(uniques)}
    cat_cardinalities.append(len(uniques) + 1)


def encode_categoricals(d: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for col in CATEGORICAL:
        mapping = cat_maps[col]
        exprs.append(
            pl.col(col).cast(pl.Utf8).replace_strict(mapping, default=0).cast(pl.Int64).alias(f"_{col}_code")
        )
    return d.with_columns(exprs)


train_df = encode_categoricals(train_df)
test_df = encode_categoricals(test_df)
CAT_CODE_COLS = [f"_{c}_code" for c in CATEGORICAL]


In [14]:
# =====================================================================
# 3. Normalize numerics — fit mean/std on TRAIN only
# =====================================================================
num_means = {c: train_df[c].mean() for c in NUMERIC}
num_stds = {c: (train_df[c].std() or 1.0) for c in NUMERIC}
num_stds = {c: (s if s and s > 1e-8 else 1.0) for c, s in num_stds.items()}


def normalize_numeric(d: pl.DataFrame) -> pl.DataFrame:
    return d.with_columns([
        ((pl.col(c) - num_means[c]) / num_stds[c]).alias(c) for c in NUMERIC
    ])


train_df = normalize_numeric(train_df)
test_df = normalize_numeric(test_df)


In [15]:
# =====================================================================
# 4. Build one sequence (stops sorted by stop_sequence) per trip
# =====================================================================
def build_sequences(d: pl.DataFrame) -> list[dict]:
    d = d.sort(SEQ_ID_COLS + [ORDER_COL])
    sequences = []
    for _, group in d.group_by("_seq_id", maintain_order=True):
        cat = group.select(CAT_CODE_COLS).to_numpy()
        num = group.select(NUMERIC).to_numpy().astype(np.float32)
        target = group[TARGET].to_numpy().astype(np.float32)
        sequences.append({"cat": cat, "num": num, "target": target, "length": len(target)})
    return sequences


print("Building train sequences...")
train_sequences = build_sequences(train_df)
print("Building test sequences...")
test_sequences = build_sequences(test_df)
print(f"train sequences: {len(train_sequences):,} | test sequences: {len(test_sequences):,}")

max_len_seen = max(max(s["length"] for s in train_sequences), max(s["length"] for s in test_sequences))
print(f"longest trip: {max_len_seen} stops (MAX_SEQ_LEN = {MAX_SEQ_LEN})")
assert max_len_seen <= MAX_SEQ_LEN, "increase MAX_SEQ_LEN above — a trip exceeds the positional encoding table size"


Building train sequences...
Building test sequences...
train sequences: 96,308 | test sequences: 24,077
longest trip: 103 stops (MAX_SEQ_LEN = 128)


In [16]:
# =====================================================================
# 5. Dataset / collate — pad each batch to its own max length
# =====================================================================
class TripSequenceDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        s = self.sequences[idx]
        return (
            torch.tensor(s["cat"], dtype=torch.long),
            torch.tensor(s["num"], dtype=torch.float32),
            torch.tensor(s["target"], dtype=torch.float32),
            s["length"],
        )


def collate(batch):
    cats, nums, targets, lengths = zip(*batch)
    lengths = torch.tensor(lengths, dtype=torch.long)
    cats_padded = pad_sequence(cats, batch_first=True, padding_value=0)
    nums_padded = pad_sequence(nums, batch_first=True, padding_value=0.0)
    targets_padded = pad_sequence(targets, batch_first=True, padding_value=0.0)
    return cats_padded, nums_padded, targets_padded, lengths


train_loader = DataLoader(TripSequenceDataset(train_sequences), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
test_loader = DataLoader(TripSequenceDataset(test_sequences), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)


## 6. Model — non-causal Transformer encoder

A standard sinusoidal positional encoding (Vaswani et al., 2017) is added after projecting the
concatenated embeddings + numeric features to `D_MODEL`, since self-attention itself has no
notion of stop order — without it, shuffling the stops within a trip wouldn't change the output
at all.

In [17]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model), not a learned parameter

    def forward(self, x):
        # x: (B, T, d_model)
        return self.dropout(x + self.pe[:, : x.size(1)])


class TransformerTravelTime(nn.Module):
    def __init__(self, cat_cardinalities, n_numeric, d_model, n_heads, n_layers,
                 dim_feedforward, dropout, max_len):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        # same fast.ai dynamic embedding-sizing rule as the baseline notebooks
        emb_dims = [min(EMB_DIM_CAP, max(1, int(1.6 * (c ** 0.56)))) for c in cat_cardinalities]
        self.embeddings = nn.ModuleList([
            nn.Embedding(c, d, padding_idx=0) for c, d in zip(cat_cardinalities, emb_dims)
        ])
        in_dim = sum(emb_dims) + n_numeric
        self.input_proj = nn.Linear(in_dim, d_model)
        self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_len, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
            dropout=dropout, activation="relu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, cat, num, lengths):
        B, T, _ = num.shape
        emb = [e(cat[:, :, i]) for i, e in enumerate(self.embeddings)]
        x = torch.cat(emb + [num], dim=-1)          # (B, T, in_dim)
        x = self.input_proj(x)                      # (B, T, d_model)
        x = self.pos_encoding(x)

        # padding mask only — no causal mask, every stop can attend to every other stop
        key_padding_mask = torch.arange(T, device=num.device).unsqueeze(0) >= lengths.unsqueeze(1).to(num.device)
        x = self.encoder(x, src_key_padding_mask=key_padding_mask)

        return self.head(x).squeeze(-1)


def masked_mae(pred: torch.Tensor, target: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    mask = (torch.arange(pred.size(1), device=pred.device).unsqueeze(0) < lengths.unsqueeze(1).to(pred.device))
    return (torch.abs(pred - target) * mask).sum() / mask.sum().clamp(min=1)


def masked_mse(pred: torch.Tensor, target: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    mask = (torch.arange(pred.size(1), device=pred.device).unsqueeze(0) < lengths.unsqueeze(1).to(pred.device))
    return (((pred - target) ** 2) * mask).sum() / mask.sum().clamp(min=1)


model = TransformerTravelTime(
    cat_cardinalities, len(NUMERIC), D_MODEL, N_HEADS, N_LAYERS,
    DIM_FEEDFORWARD, DROPOUT, MAX_SEQ_LEN,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-3, epochs=MAX_EPOCHS, steps_per_epoch=len(train_loader),
)
print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {n_params:,}")


TransformerTravelTime(
  (embeddings): ModuleList(
    (0): Embedding(6, 4, padding_idx=0)
    (1): Embedding(3, 2, padding_idx=0)
    (2): Embedding(79, 18, padding_idx=0)
    (3): Embedding(49, 14, padding_idx=0)
    (4-5): 2 x Embedding(3, 2, padding_idx=0)
    (6): Embedding(2, 2, padding_idx=0)
    (7-11): 5 x Embedding(3, 2, padding_idx=0)
    (12): Embedding(14, 7, padding_idx=0)
  )
  (input_proj): Linear(in_features=84, out_features=256, bias=True)
  (pos_encoding): SinusoidalPositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=Tr

C:\Users\ishan\AppData\Local\Temp\ipykernel_7764\1285289055.py:36: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


In [18]:
# =====================================================================
# 7. Train loop with early stopping on validation (test) MAE
# =====================================================================
def run_epoch(loader, train: bool) -> float:
    model.train() if train else model.eval()
    total_mae, total_count = 0.0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for cat, num, target, lengths in loader:
            cat, num, target = cat.to(DEVICE), num.to(DEVICE), target.to(DEVICE)
            pred = model(cat, num, lengths)
            loss = masked_mse(pred, target, lengths)
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
                scheduler.step()
            mae = masked_mae(pred, target, lengths)
            n = lengths.sum().item()
            total_mae += mae.item() * n
            total_count += n
    return total_mae / total_count


best_val_mae = float("inf")
epochs_no_improve = 0
best_state = None

for epoch in range(1, MAX_EPOCHS + 1):
    train_mae = run_epoch(train_loader, train=True)
    val_mae = run_epoch(test_loader, train=False)
    print(f"epoch {epoch:3d} | train MAE {train_mae:7.2f}s | val MAE {val_mae:7.2f}s")

    if val_mae < best_val_mae - 1e-3:
        best_val_mae = val_mae
        epochs_no_improve = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (best val MAE {best_val_mae:.2f}s)")
            break

model.load_state_dict(best_state)


epoch   1 | train MAE   36.81s | val MAE   33.25s
epoch   2 | train MAE   33.38s | val MAE   32.68s
epoch   3 | train MAE   32.80s | val MAE   32.14s
epoch   4 | train MAE   32.44s | val MAE   31.69s
epoch   5 | train MAE   32.20s | val MAE   31.80s
epoch   6 | train MAE   32.01s | val MAE   31.83s
epoch   7 | train MAE   31.93s | val MAE   32.19s
epoch   8 | train MAE   31.87s | val MAE   31.52s
epoch   9 | train MAE   31.81s | val MAE   32.31s
epoch  10 | train MAE   31.75s | val MAE   31.21s
epoch  11 | train MAE   31.65s | val MAE   30.66s
epoch  12 | train MAE   31.50s | val MAE   31.33s
epoch  13 | train MAE   31.22s | val MAE   29.91s
epoch  14 | train MAE   30.89s | val MAE   28.97s
epoch  15 | train MAE   30.35s | val MAE   28.22s
epoch  16 | train MAE   30.06s | val MAE   27.98s
epoch  17 | train MAE   29.81s | val MAE   27.81s
epoch  18 | train MAE   29.69s | val MAE   27.76s
epoch  19 | train MAE   29.55s | val MAE   27.57s
epoch  20 | train MAE   29.49s | val MAE   27.57s


<All keys matched successfully>

In [19]:
# =====================================================================
# 8. Final evaluation — same metrics as the other notebooks, directly comparable
#
# NOTE: because this model is non-causal, these numbers represent an offline / "sees the
# whole trip" upper bound, not a real-time ETA estimate. Compare against the LSTM /
# Conv-Attn-LSTM / TFT numbers with that in mind, not as a strictly apples-to-apples
# real-time comparison.
# =====================================================================
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for cat, num, target, lengths in test_loader:
        cat, num = cat.to(DEVICE), num.to(DEVICE)
        pred = model(cat, num, lengths).cpu()
        for i, length in enumerate(lengths):
            all_preds.append(pred[i, :length].numpy())
            all_targets.append(target[i, :length].numpy())

preds = np.concatenate(all_preds)
targets = np.concatenate(all_targets)
resid = preds - targets

mae = np.abs(resid).mean()
rmse = np.sqrt((resid ** 2).mean())
median_ae = np.median(np.abs(resid))
bias = resid.mean()
ss_res = (resid ** 2).sum()
ss_tot = ((targets - targets.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot

print(f"\nMAE:       {mae:.1f} sec")
print(f"Median AE: {median_ae:.1f} sec")
print(f"RMSE:      {rmse:.1f} sec")
print(f"Bias:      {bias:+.1f} sec")
print(f"R2:        {r2:.4f}")
for thresh in (30, 60, 120):
    print(f"within {thresh}s: {(np.abs(resid) <= thresh).mean():.1%}")

import os
os.makedirs("models", exist_ok=True)
torch.save(
    {
        "model_state": model.state_dict(),
        "cat_maps": cat_maps,
        "num_means": num_means,
        "num_stds": num_stds,
        "config": {
            "D_MODEL": D_MODEL, "N_HEADS": N_HEADS, "N_LAYERS": N_LAYERS,
            "DIM_FEEDFORWARD": DIM_FEEDFORWARD, "DROPOUT": DROPOUT, "MAX_SEQ_LEN": MAX_SEQ_LEN,
            "NUMERIC": NUMERIC, "CATEGORICAL": CATEGORICAL,
        },
    },
    "models/transformer_noncausal.pt",
)
print("\nmodel saved to models/transformer_noncausal.pt")



MAE:       24.4 sec
Median AE: 16.5 sec
RMSE:      47.6 sec
Bias:      -3.8 sec
R2:        0.6212
within 30s: 74.5%
within 60s: 93.9%
within 120s: 98.7%

model saved to models/transformer_noncausal.pt
